In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: fixed CONT_FINAL six-bit derivative diagnostic

Run all measures only fixed bits **[4,12,18,20,27,30]** or resumes unfinished units.
The input is the actual `terminal` field of Continue/resumable-task-v1-2x010/run/step_2_unit.pt.
The old gradient checkpoint's input_terminal is MID and must not be used.
Original BASE comes from that run's inputs/terminal_latent.pt; saved FINAL readouts
come from result.json. Input paths and SOURCE_REF are editable locators.

Stable independent output:
MyDrive/Video-WM/Local-Joint-State-Payload-V1-Six-Bit-Final/resumable-six-bit-final-v1/.
Keep this folder when reopening. Each Run all appends attempt logs; complete paired
gradient units are skipped. All six saved means CPU report-only with zero VAE calls.
Missing/corrupt state or invalid committed input/bit pairs is reported explicitly,
never silently treated as permission to repeat the full experiment.

Uninterrupted nominal budget: **1 load + 6 gradient decode + 6 VJP**, six independent
fresh graphs in serial; **276 forward + 276 nominal recompute chunks**.
Ordinary decode/encode/DiT/native/scheduler/codec: **0**. No shared-graph multi-backward.
Each forward retains **55 quantities: 32 margins,22 correlations,G; 88 windows/1408 chips**,
payload denominators [24,24,20,20]. Across six rows:528 windows/8448 chips/330 metrics;
repeated same-point measurements are not six independent samples. Failed/missing rows remain.

Differentiate the positive margin exactly at flat index i. Mask once and retain the
unnormalized h_i. Compare each 55-value readout to saved FINAL; differences are diagnostics.
Original key/message/mask/rho/cap/readout remain. The full scalar loss gradient equals
-0.025 sum(h_i) only if payload_active is exactly the six bits, all six remain negative,
state_active is false, and coefficients apply on every read. Fragment-min and wrong-offset
branches are not this scalar-loss active set. Otherwise retain derivatives without that claim.

CPU reports raw Gram/cosines/norms, aggregate slopes, radial/tangential decomposition,
and the unnormalized simplex minimum-norm common directions, with primal/dual/KKT
residuals and FP64 error bounds. Strict common direction is numerically supported only
if all six slope lower bounds remain positive. Otherwise NUMERICALLY_UNRESOLVED;
a near-zero solution or negative cosine does not establish infeasibility.
Pure tangent uses u=(FINAL-BASE)/r, t_i=h_i-(h_i dot u)u. Ball distances are geometry,
not gain predictions or steps to cross zero. Numerical bounds concern stored gradients,
not the unknown error of the FP32 model derivative.

No update/candidate/scan/rollback/cap increase/COMMON refresh/M1/M2. Other26 positive-bit,
state and quality derivatives are unmeasured; no protection or finite-budget reachability
claim. This known-message local diagnostic is not scientific PASS or route infeasibility.

Inputs are backed up before model load. Each complete gradient/readout/input/bit unit
commits immediately; every60 seconds saves the last committed state/progress and retains
the previous readable fallback. In-flight graphs restart only as whole unfinished units.
Activation spool is temporary, not a cross-session autograd checkpoint. Cumulative physical
calls, failures and current-attempt counts are separate; interruption can exceed nominal cost.

Reuses the successful Wan FP32 causal decoder/checkpoint/spool path. Historical single-unit
spool peak ~64.53GiB; serial six units reuse capacity, while total per-direction I/O is ~387.2GiB.
Historical GPU peak allocated19.01GiB/reserved21.72GiB; approximate six-unit time44 minutes
plus load/readout/save. These are references, not guarantees or GPU model gates.
Validation is CPU/fake/static; real GPU/Colab execution is user-run. Return the entire folder.

In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable GitHub branch/tag/ref.
FIXED_CONFIG = {'schema': 'local-joint-six-bit-final-v1', 'source_run': 'resumable-task-v1-2x010', 'source_arm': 'CONT_FINAL', 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'revision': '0fad780a534b6463e45facd96134c9f345acfa5b'}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5, 'cap': 1.0}, 'device': 'cuda', 'inputs': {'terminal_latent': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit-Continue/resumable-task-v1-2x010/run/inputs/terminal_latent.pt', 'final_step_unit': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit-Continue/resumable-task-v1-2x010/run/step_2_unit.pt', 'saved_final_result': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit-Continue/resumable-task-v1-2x010/run/result.json'}, 'bits': [4, 12, 18, 20, 27, 30], 'loss': {'mu': 0.02, 'gamma': 0.01, 'payload_weight': 0.8, 'state_weight': 0.2}}
EXPECTED_COST = {'vae_load': 1, 'decode': 6, 'decoder_vjp': 6, 'ordinary_decode': 0, 'encode': 0, 'dit': 0, 'native': 0, 'scheduler': 0, 'codec': 0, 'ordinary_chunks': 0, 'gradient_forward_chunks': 276, 'nominal_replay_chunks': 276, 'windows': 528, 'chips': 8448, 'metrics': 330}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Six-Bit-Final')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / 'resumable-six-bit-final-v1'  # Same logical task on every Run all.
OUTPUT.mkdir(exist_ok=True)
ATTEMPT_OUTPUT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT_OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-Six-Bit-Final-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(ATTEMPT_OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (ATTEMPT_OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_six_bit_final_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    views={name: dict(status='NOT_RUN', expected_windows=88, expected_chips=1408, expected_metrics=55)
           for name in ('bit_04', 'bit_12', 'bit_18', 'bit_20', 'bit_27', 'bit_30')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
write_json(ATTEMPT_OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Fixed SIX-BIT-FINAL output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_six_bit_final_v1_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    # A repeated Run all may retain packages from an earlier extracted workspace.
    import importlib
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(ATTEMPT_OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
from runtime.wan.local_joint_six_bit_final_v1 import model_work_remaining
try:
    NEEDS_MODEL = model_work_remaining(RUN_OUTPUT)
    probe_path = OUTPUT / 'dependency_probe.json'
    probe_lines = [
        "import json, pathlib, sys, torch, numpy, PIL",
        "row = dict(python=sys.version, torch=str(torch.__version__), model_work_remaining=" + repr(NEEDS_MODEL) + ")",
    ]
    if NEEDS_MODEL:
        probe_lines += ["from diffusers import AutoencoderKLWan"]
        if str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
            probe_lines += ["assert torch.cuda.is_available(), 'CUDA is needed for unfinished decode/VJP'",
                            "row.update(cuda_available=True, device=torch.cuda.get_device_name(0))"]
    probe_lines += ["pathlib.Path(" + repr(str(probe_path)) + ").write_text(json.dumps(row))"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        packages = ['torch', 'numpy', 'pillow']
        if NEEDS_MODEL:
            packages += ['diffusers', 'transformers', 'accelerate', 'safetensors', 'huggingface-hub', 'sentencepiece', 'ftfy']
        logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    environment_receipt = json.loads(probe_path.read_text())
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode,
        source_and_versions_are_location_notes=True, checkpoint_interval_seconds=60)
    write_json(ATTEMPT_OUTPUT / 'environment_receipt.json', environment_receipt)
    print('Remaining model work:', NEEDS_MODEL, '; task:', OUTPUT, flush=True)
except BaseException as exc:
    record_failure('ENVIRONMENT_SETUP', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_six_bit_final_v1_run',
               '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_SIX_BIT_FINAL', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_six_bit_final_v1 import finalize_interrupted
    # Takeover is report-only and follows child/process-group reap in logged().
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.exists() else None
    write_json(ATTEMPT_OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] != 'COMPLETE':
        raise RuntimeError('SIX-BIT-FINAL engineering interruption; retain entire output: ' + str(OUTPUT))
    fixed_slots.update(status='SUPERSEDED_BY_RUN_RESULT', result='run/result.json')
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_SIX_BIT_FINAL', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
print('Engineering status:', result['status'])
print('Nominal budget:', result['expected_cost'])
print('Cumulative physical calls:', result['counts'])
print('This Run all:', result.get('current_attempt_counts', {}))
print('Fixed totals:', result.get('totals'))
for name, row in result['views'].items():
    print(name, row['status'], row.get('gradient_norms'), row.get('reason'))
for name, replay in result['replay'].items():
    print(name, replay['counts'], replay.get('boundary_storage', {}))
diagnostic_path = RUN_OUTPUT / 'diagnostic.json'
if diagnostic_path.exists():
    diagnostic = json.loads(diagnostic_path.read_text())
    print('Full current scalar-loss gradient formula valid:', diagnostic['loss_gradient_formula_valid_on_all_six_reads'])
    print('Common:', diagnostic['common']['status'], 'Tangent:', diagnostic['tangent']['status'])
    print('Numerical scope:', diagnostic['numerical_error_scope'])
print(result['ceiling'])
print('Return entire output folder:', OUTPUT)